# 03 · Reconstructing midterm scores from the log

The gradebook isn't in the data, so each student's midterm score is rebuilt from the answers they logged
inside their exam window. Rules per question type:

| type | rule |
|---|---|
| multiple choice | last answer submitted, correct/incorrect as logged |
| Parsons | best attempt; opened but never submitted = 0 |
| ActiveCode | best unit-test pass rate; never run = 0 |
| fill-in / clickable / drag-and-drop | compared to an **inferred** answer key (see below) |
| short answer | not scoreable, excluded and reported |

In [1]:
import pandas as pd
from so_ell import config
from so_ell.notebook import show, show_sql, describe
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config.ensure_dirs()
T = config.TABLES_DIR

## 1. Parsing the logged answers

In [2]:
from so_ell import scoring, scores
show(scoring.activity_family, scoring.parse_mchoice_score, scoring.parse_parsons_score, scoring.parse_unittest_pct, scoring.normalize_response)

`so_ell.scoring.activity_family`
```python
def activity_family(selection: object) -> str:
    mapping = {
        "mChoice": "mchoice",
        "parsonsMove": "parsons",
        "parsons": "parsons",
        "hparsons": "parsons",
        "hparsonsAnswer": "parsons",
        "activecode": "activecode",
        "unittest": "activecode",
        "ac_error": "activecode",
        "fillb": "fillb",
        "clickableArea": "clickable",
        "dragNdrop": "dragndrop",
        "dragNdrop-drop": "dragndrop",
        "shortanswer": "shortanswer",
        "selectquestion": "conceptcheck",
    }
    return mapping.get(str(selection), "other")
```

`so_ell.scoring.parse_mchoice_score`
```python
def parse_mchoice_score(action: object) -> float:
    text = str(action)
    if ":correct" in text:
        return 1.0
    if ":no" in text:
        return 0.0
    return np.nan
```

`so_ell.scoring.parse_parsons_score`
```python
def parse_parsons_score(action: object) -> float:
    # Regular Parsons checks look like "correct:..." / "incorrect:...".
    text = str(action)
    if text.startswith("correct"):
        return 1.0
    if text.startswith("incorrect"):
        return 0.0
    return np.nan
```

`so_ell.scoring.parse_unittest_pct`
```python
def parse_unittest_pct(action: object) -> float:
    # ActiveCode stores something like "percent:80.0" when unit tests run.
    match = re.search(r"percent:([0-9.]+)", str(action))
    return float(match.group(1)) / 100 if match else np.nan
```

`so_ell.scoring.normalize_response`
```python
def normalize_response(value: object) -> str | None:
    """Smash fill-in / click / drag answers into a comparable string."""
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    parsed = None
    try:
        parsed = ast.literal_eval(text)
    except Exception:
        parsed = None
    value_to_use = parsed if parsed is not None else text
    if isinstance(value_to_use, dict):
        normalized = {}
        for key, val in sorted(value_to_use.items()):
            if isinstance(val, list):
                normalized[str(key).strip().lower()] = sorted(str(item).strip().lower() for item in val)
            else:
                normalized[str(key).strip().lower()] = str(val).strip().lower()
        return json.dumps(normalized, sort_keys=True)
    if isinstance(value_to_use, (list, tuple, set)):
        return json.dumps([str(item).strip().lower() for item in value_to_use])
    compact = re.sub(r"\s+", " ", str(value_to_use).strip().strip("\"'").lower())
    return compact or None
```

## 2. Load only the answer events
Page views and navigation are skipped; only answer selections are read, and the large text columns are kept only where a parser needs them.

In [3]:
show(scores.load_scoreable_events)
raw = scores.load_scoreable_events()
print(f"{len(raw):,} answer events; {raw.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")
raw["selection"].value_counts().to_frame("events")

`so_ell.scores.load_scoreable_events`
```python
def load_scoreable_events(events_path: Path = config.EVENTS_PARQUET) -> pd.DataFrame:
    """Only the answer-type events; page views etc. never feed a score."""
    con = duckdb.connect()
    con.execute("SET enable_progress_bar = false")
    selections = ", ".join(f"'{s}'" for s in EXAM_SELECTIONS)
    semesters = ", ".join(f"'{s}'" for s in config.ANALYSIS_SEMESTERS)
    parsed = ", ".join(f"'{s}'" for s in PARSED_SELECTIONS)
    keyed = ", ".join(f"'{s}'" for s in KEYED_SELECTIONS)
    raw = con.execute(
        f"""
        SELECT time, anon_student_id, problem_name, session_id, selection, semester AS semester_raw,
               -- action / input text is only parsed for these selections; dropping it
               -- elsewhere keeps 16M rows in a few GB instead of ~10.
               CASE WHEN selection IN ({parsed}) THEN action END AS action,
               CASE WHEN selection IN ({keyed}) THEN input END AS input
        FROM read_parquet('{events_path.as_posix()}')
        WHERE selection IN ({selections}) AND semester IN ({semesters}) AND time IS NOT NULL
        """
    ).df()
    con.close()
    for col in ["anon_student_id", "session_id", "selection", "semester_raw"]:
        raw[col] = raw[col].astype("category")
    return raw
```

16,764,045 answer events; 1.1 GB in memory


,events
selection,
activecode,5739917
selectquestion,4500902
parsonsMove,2516498
mChoice,1005422
parsons,737139
unittest,620668
fillb,513041
ac_error,489071
hparsons,320427


## 3. Exam windows and per-event scores

In [4]:
show(scores.build_windows, scores.add_event_scores, scores.label_exam_events)
windows = scores.build_windows(pd.read_parquet(config.EXAM_ATTEMPTS_PARQUET))
print(f"{len(windows):,} student-exam windows")

`so_ell.scores.build_windows`
```python
def build_windows(attempts: pd.DataFrame) -> pd.DataFrame:
    """Collapse a student's attempts (e.g. a crashed and resumed session) into one window."""
    return (
        attempts.groupby(WINDOW_KEYS + ["semester", "normalized_midterm_name"], as_index=False, observed=True)
        .agg(
            exam_start=("attempt_start", "min"),
            exam_end=("attempt_end", "max"),
            n_sessions=("session_id", "nunique"),
            total_exam_minutes=("attempt_minutes", "sum"),
        )
        .sort_values(WINDOW_KEYS)
        .reset_index(drop=True)
    )
```

`so_ell.scores.add_event_scores`
```python
def add_event_scores(events: pd.DataFrame) -> pd.DataFrame:
    events = events.copy()
    events["family"] = events["selection"].astype(str).map(activity_family)
    events["event_score"] = np.nan
    parsers = {
        "mChoice": parse_mchoice_score,
        "parsons": parse_parsons_score,
        "hparsonsAnswer": parse_hparsons_score,
        "unittest": parse_unittest_pct,
    }
    for selection, parser in parsers.items():
        mask = events["selection"].eq(selection)
        if mask.any():
            events.loc[mask, "event_score"] = events.loc[mask, "action"].map(parser).astype(float)
    keyed = events["family"].isin(KEYED_FAMILIES)
    source = events["input"].where(events["input"].notna(), events["action"])
    events["response_norm"] = pd.Series(None, index=events.index, dtype="object")
    if keyed.any():
        events.loc[keyed, "response_norm"] = source[keyed].map(normalize_response)
    return events
```

`so_ell.scores.label_exam_events`
```python
def label_exam_events(events: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:
    labeled = events.merge(
        windows[WINDOW_KEYS + ["semester", "normalized_midterm_name", "exam_start", "exam_end"]],
        on=["semester_raw", "anon_student_id"],
        how="inner",
    )
    inside = labeled["time"].between(labeled["exam_start"], labeled["exam_end"], inclusive="both")
    return labeled.loc[inside].copy()
```

2,780 student-exam windows


## 4. Inferred answer keys

Fill-in, clickable and drag-and-drop events don't carry a correct/incorrect flag. For each problem the key is the
most common answer among students who reached a final answer, used only when at least 65% agree
(`KEY_CONFIDENCE_THRESHOLD`). Notebook 05 re-runs the headline result at thresholds from 55% to "no inferred keys".

In [5]:
show(scores.infer_answer_keys)
print("threshold:", config.KEY_CONFIDENCE_THRESHOLD)

`so_ell.scores.infer_answer_keys`
```python
def infer_answer_keys(events: pd.DataFrame) -> pd.DataFrame:
    """Modal final answer per problem, with the share of students who gave it."""
    cols = ["problem_name", "answer_key", "key_n", "key_support", "key_confidence"]
    keyed = events.loc[events["family"].isin(KEYED_FAMILIES) & events["response_norm"].notna()]
    if keyed.empty:
        return pd.DataFrame(columns=cols)
    final = (
        keyed.sort_values(["problem_name", "anon_student_id", "time"])
        .groupby(["problem_name", "anon_student_id"], observed=True)
        .tail(1)
    )
    counts = final.groupby(["problem_name", "response_norm"], as_index=False).size().rename(columns={"size": "key_support"})
    totals = final.groupby("problem_name", as_index=False).size().rename(columns={"size": "key_n"})
    top = (
        counts.sort_values(["problem_name", "key_support", "response_norm"], ascending=[True, False, True])
        .groupby("problem_name")
        .head(1)
        .rename(columns={"response_norm": "answer_key"})
    )
    lookup = top.merge(totals, on="problem_name")
    lookup["key_confidence"] = lookup["key_support"] / lookup["key_n"]
    return lookup[cols]
```

threshold: 0.65


## 5. Score every student-exam

In [6]:
show(scores._score_problem, scores.score_exams)
result = scores.score_exams(raw, windows)
from so_ell import validation
validation.validate(result["student_scores"], validation.student_scores_schema)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
result["problem_scores"].to_parquet(config.PROCESSED_DIR / "student_exam_problem_scores.parquet", index=False)
result["student_scores"].to_parquet(config.PROCESSED_DIR / "student_exam_scores.parquet", index=False)
result["semester_summary"].to_csv(T / "cohort_score_summary.csv", index=False)
result["answer_keys"].drop(columns=["answer_key"]).to_csv(T / "inferred_answer_keys.csv", index=False)
result["unsupported_summary"].to_csv(T / "unscored_items_summary.csv", index=False)
print(f"{len(result['student_scores']):,} student-exam scores, "
      f"{result['student_scores']['anon_student_id'].nunique():,} students")

`so_ell.scores._score_problem`
```python
def _score_problem(group: pd.DataFrame, key_threshold: float) -> dict[str, object]:
    family = group["family"].mode().iloc[0]
    score, source, key_conf = np.nan, "unscored", np.nan
    values = group["event_score"].dropna()
    if family == "mchoice":
        if not values.empty:
            score, source = values.iloc[-1], "last_mchoice_result"
    elif family == "parsons":
        score, source = (values.max(), "best_parsons_result") if not values.empty else (0.0, "parsons_no_success")
    elif family == "activecode":
        tests = group.loc[group["selection"].eq("unittest"), "event_score"].dropna()
        score, source = (tests.max(), "best_unittest_percent") if not tests.empty else (0.0, "activecode_no_passing_test")
    elif family in KEYED_FAMILIES:
        finals = group.loc[group["response_norm"].notna()]
        if finals.empty:
            source = f"{family}_no_response"
        else:
            last = finals.iloc[-1]
            key_conf = last["key_confidence"]
            if pd.notna(last["answer_key"]) and pd.notna(key_conf) and key_conf >= key_threshold:
                score, source = float(last["response_norm"] == last["answer_key"]), f"{family}_modal_key"
            else:
                source = f"{family}_low_confidence_key"
    elif family == "shortanswer":
        source = "unsupported_shortanswer"
    return {
        "item_family": family,
        "score": score,
        "score_source": source,
        "key_confidence": key_conf,
        "n_events": len(group),
    }
```

`so_ell.scores.score_exams`
```python
def score_exams(
    raw: pd.DataFrame,
    windows: pd.DataFrame,
    key_threshold: float = config.KEY_CONFIDENCE_THRESHOLD,
) -> dict[str, pd.DataFrame]:
    exam_events = add_event_scores(raw.loc[raw["selection"].isin(SCORED_SELECTIONS)])
    exam_events = label_exam_events(exam_events, windows)
    keys = infer_answer_keys(exam_events)
    exam_events = exam_events.merge(keys, on="problem_name", how="left")

    group_cols = WINDOW_KEYS + ["semester", "normalized_midterm_name", "problem_name"]
    exam_events = exam_events.sort_values(group_cols + ["time"])
    rows = []
    for key, group in exam_events.groupby(group_cols, observed=True, sort=False):
        rows.append(dict(zip(group_cols, key, strict=True)) | _score_problem(group, key_threshold))
    problem_scores = pd.DataFrame(rows)

    student_scores = (
        problem_scores.groupby(WINDOW_KEYS + ["semester", "normalized_midterm_name"], as_index=False, observed=True)
        .agg(
            n_items_observed=("problem_name", "nunique"),
            n_items_scored=("score", lambda s: s.notna().sum()),
            score_points=("score", "sum"),
            score_pct=("score", "mean"),
        )
        .merge(windows, on=WINDOW_KEYS + ["semester", "normalized_midterm_name"], how="left")
    )
    student_scores["score_pct"] *= 100
    student_scores["coverage_pct"] = 100 * student_scores["n_items_scored"] / student_scores["n_items_observed"]
    student_scores = student_scores.loc[student_scores["score_pct"].notna()].reset_index(drop=True)

    semester_summary = (
        student_scores.groupby(["semester_raw", "semester", "midterm"], as_index=False)
        .agg(
            n_students=("anon_student_id", "nunique"),
            mean_score_pct=("score_pct", "mean"),
            median_score_pct=("score_pct", "median"),
            sd_score_pct=("score_pct", "std"),
            mean_coverage_pct=("coverage_pct", "mean"),
        )
        .sort_values(["semester_raw", "midterm"])
    )
    unsupported = problem_scores.loc[
        problem_scores["score"].isna(),
        ["semester_raw", "midterm", "problem_name", "item_family", "score_source", "key_confidence", "n_events"],
    ]
    unsupported_summary = (
        unsupported.groupby(["item_family", "score_source"], as_index=False)
        .agg(student_problem_rows=("problem_name", "size"), distinct_problems=("problem_name", "nunique"))
        .sort_values("student_problem_rows", ascending=False)
    )
    return {
        "problem_scores": problem_scores,
        "student_scores": student_scores,
        "semester_summary": semester_summary,
        "answer_keys": keys.sort_values(["key_confidence", "key_n"], ascending=[True, False]),
        "unsupported_summary": unsupported_summary,
    }
```

2,745 student-exam scores, 1,397 students


## 6. Results by cohort
Exam difficulty moves a lot between semesters, which is why every later comparison is made *within* a semester × midterm cohort.

In [7]:
result["semester_summary"].round(2)

,semester_raw,semester,midterm,n_students,mean_score_pct,median_score_pct,sd_score_pct,mean_coverage_pct
0,F21,Fall 2021,mid1,103,78.34,82.67,11.72,99.99
1,F21,Fall 2021,mid2,102,65.22,66.00,13.32,86.32
2,F22,Fall 2022,mid1,145,78.63,81.82,12.59,100.00
3,F22,Fall 2022,mid2,145,65.72,68.62,13.61,90.61
4,F23,Fall 2023,mid1,205,71.43,75.67,16.70,96.77
5,F23,Fall 2023,mid2,199,66.89,70.00,12.44,90.92
6,F24,Fall 2024,mid1,219,65.81,66.43,18.59,96.70
7,F24,Fall 2024,mid2,220,61.62,63.67,14.42,100.00
8,F25,Fall 2025,mid1,213,69.90,72.41,20.02,96.66
9,F25,Fall 2025,mid2,211,65.10,65.52,18.38,100.00


Answer-key confidence (keys themselves are not exported):

In [8]:
keys = result["answer_keys"]
pd.Series({
    "problems with a candidate key": len(keys),
    "keys used (>= threshold)": int((keys["key_confidence"] >= config.KEY_CONFIDENCE_THRESHOLD).sum()),
    "median agreement": round(float(keys["key_confidence"].median()), 3),
}).to_frame("value")

,value
problems with a candidate key,78.0
keys used (>= threshold),63.0
median agreement,1.0


Items that could not be scored:

In [9]:
result["unsupported_summary"]

,item_family,score_source,student_problem_rows,distinct_problems
0,fillb,fillb_low_confidence_key,4246,15


In [10]:
del raw